# Adaptive Inference Optimization Under Baseline Drift

### Rejecting false wins when the baseline itself moves

**Question.** How should an automated performance optimizer decide whether a candidate is truly better when cold start, compilation, caches, allocator state, and runtime warm-up can move the baseline?

This notebook intentionally does **not** claim a successful optimization from point estimates that cannot support that conclusion.


## 1 — Recorded evidence

```text
cold baseline      1375.208 tokens/s/device
hot baseline       1441.083 tokens/s/device
best candidate     1454.510 tokens/s/device
```

Relative to the cold baseline, the candidate appears **5.77%** faster. Relative to the hot baseline—the more defensible comparator after warm-up—the gain is only **0.93%**.

That difference is the experiment.


In [ ]:
from pathlib import Path
import json
recorded=json.loads(Path("../data/optimizer_recorded_run.json").read_text())
cold=recorded["baseline_cold"]; hot=recorded["baseline_hot"]; cand=recorded["candidate_best"]
print(f"candidate vs cold: {(cand/cold-1)*100:.4f}%")
print(f"candidate vs hot : {(cand/hot-1)*100:.4f}%")
print(f"baseline drift   : {(hot/cold-1)*100:.4f}%")


## 2 — Why point-estimate optimization is dangerous

The cold→hot baseline drift is larger than the candidate's improvement over the hot baseline. A controller that compares every candidate against the first baseline can therefore promote a configuration whose apparent gain is mostly runtime state.

```text
measured candidate gain
    = true configuration gain
      + baseline-state drift
      + measurement noise
```

The optimizer needs a **promotion contract**, not just `if throughput > best`.


## 3 — State machine

```text
BASELINE_STABILIZE
      ↓
PROFILE
      ↓
PROPOSE
      ↓
MEASURE
      ↓
VALIDATE
  ↙       ↘
REJECT   ACCEPT
      ↓
STOP / REPORT
```

A candidate is never promoted directly from a microbenchmark or a single serving measurement.


In [ ]:
from dataclasses import dataclass
from enum import Enum

class Decision(Enum):
    ACCEPT="ACCEPT"
    REJECT="REJECT"
    INSUFFICIENT_EVIDENCE="INSUFFICIENT_EVIDENCE"

@dataclass
class PromotionContract:
    min_gain_pct: float = 2.0
    max_quality_regression_pct: float = 0.0
    max_p99_regression_pct: float = 2.0
    min_repeats: int = 5
    require_accuracy_gate: bool = True

@dataclass
class Evidence:
    baseline_samples: list[float]
    candidate_samples: list[float]
    quality_gate_passed: bool | None
    p99_change_pct: float | None


In [ ]:
import statistics

def decide(contract: PromotionContract, e: Evidence):
    if len(e.baseline_samples) < contract.min_repeats or len(e.candidate_samples) < contract.min_repeats:
        return Decision.INSUFFICIENT_EVIDENCE, "not enough repeated measurements"
    if contract.require_accuracy_gate and e.quality_gate_passed is not True:
        return Decision.INSUFFICIENT_EVIDENCE, "quality gate missing or failed"
    if e.p99_change_pct is None:
        return Decision.INSUFFICIENT_EVIDENCE, "tail-latency evidence missing"
    if e.p99_change_pct > contract.max_p99_regression_pct:
        return Decision.REJECT, "tail-latency regression"
    b=statistics.median(e.baseline_samples); c=statistics.median(e.candidate_samples)
    gain=(c/b-1)*100
    if gain < contract.min_gain_pct:
        return Decision.REJECT, f"gain {gain:.2f}% below threshold"
    return Decision.ACCEPT, f"median gain {gain:.2f}%"


## 4 — Apply the contract

The recorded evidence contains point estimates, not repeated distributions, and the downstream accuracy evaluation did not complete. Under the contract above, the correct state is **INSUFFICIENT_EVIDENCE**, not ACCEPT.

The promotion rule is intentionally stricter than a point-estimate comparison.


In [ ]:
contract=PromotionContract(min_gain_pct=2.0,min_repeats=5)
evidence=Evidence(baseline_samples=[recorded["baseline_hot"]], candidate_samples=[recorded["candidate_best"]], quality_gate_passed=None, p99_change_pct=None)
print(decide(contract,evidence))


## 5 — Profile information guides search, not promotion

The recorded profile characterized the dominant operation as approximately **88.41%** of measured time and memory-bound, with the measured system at **68.5%** of its estimated roofline. That is useful for choosing where to search, but not proof that a proposed change improves the end-to-end workload.

```text
profile → search direction
end-to-end repeated benchmark → promotion authority
```


## 6 — Durable measurement record

Every candidate measurement should be durable and independently inspectable.


In [ ]:
from datetime import datetime, timezone
import hashlib

def canonical_hash(obj) -> str:
    raw=json.dumps(obj,sort_keys=True,separators=(",",":"),default=str).encode()
    return hashlib.sha256(raw).hexdigest()

def measurement_record(config, metrics, environment, run_id):
    payload={"run_id":run_id,"observed_at":datetime.now(timezone.utc).isoformat(),"config":config,"metrics":metrics,"environment":environment}
    payload["evidence_hash"]=canonical_hash(payload)
    return payload


## 7 — What this notebook establishes

It does **not** establish that the recorded candidate is a production win.

> **An autonomous optimizer must treat baseline stabilization, measurement repetition, quality gates, and end-to-end validation as part of the optimization algorithm itself.**

A system that cannot distinguish a 5.77% cold-baseline ‘win’ from a 0.93% hot-baseline delta is not yet optimizing reliably.
